In [1]:
import tensorflow as tf
from tensorflow import keras
import cv2
import numpy as np

In [2]:
image = cv2.imread('Tree-256x256.png')
image.shape

(256, 256, 3)

#### Patch Encoder

In [6]:
patches = tf.image.extract_patches(images=tf.expand_dims(image, axis=0),
                           sizes=[1, 16, 16, 1],
                           strides=[1, 16, 16, 1],
                           rates=[1, 1, 1, 1],
                           padding='VALID')
patches = tf.reshape(patches, (patches.shape[0], -1, patches.shape[-1]))
patches.shape

TensorShape([1, 256, 768])

In [23]:
class PatchEncoder(keras.layers.Layer):
    def __init__(self, patch_size, hidden_size):
        super(PatchEncoder, self).__init__(name='patch_encoder')
        self.patch_size = patch_size
        self.linear_projection = keras.layers.Dense(hidden_size)
        self.embedding = keras.layers.Embedding(256, hidden_size)

    def call(self, x):
        patches = tf.image.extract_patches(images=x,
                           sizes=[1, self.patch_size, self.patch_size, 1],
                           strides=[1, self.patch_size, self.patch_size, 1],
                           rates=[1, 1, 1, 1],
                           padding='VALID')
        patches = tf.reshape(patches, (patches.shape[0], -1, patches.shape[-1]))
        range_patches = tf.range(start=0, limit=patches.shape[1], delta=1)
        output = self.linear_projection(patches) + self.embedding(range_patches)
        return output


In [19]:
patch_encoder = PatchEncoder(16, 768)
patch_encoder(tf.zeros([1, 256, 256, 3]))

<tf.Tensor: shape=(1, 256, 768), dtype=float32, numpy=
array([[[-0.01762808, -0.04114603, -0.04554281, ..., -0.01347759,
         -0.00042316,  0.04889934],
        [ 0.02063699,  0.0064969 ,  0.03533555, ..., -0.03856279,
          0.04669306, -0.00233624],
        [ 0.02601005, -0.02247481, -0.01864505, ..., -0.01018973,
         -0.00251734, -0.02283056],
        ...,
        [ 0.00320269, -0.00864955, -0.03701048, ...,  0.02299504,
          0.02773366,  0.04040602],
        [ 0.03474364,  0.04085162, -0.0353673 , ..., -0.0456642 ,
         -0.01705539,  0.01848781],
        [ 0.02103059, -0.02240345, -0.04828174, ..., -0.01936752,
         -0.02602559,  0.01009788]]], dtype=float32)>

#### Transformer Encoder

In [22]:
class TransfomerEncoder(keras.layers.Layer):
    def __init__(self, n_heads, hidden_size):
      super(TransfomerEncoder, self).__init__(name="transformer_encoder")
      self.norm1 = keras.layers.LayerNormalization()
      self.norm2 = keras.layers.LayerNormalization()
      self.mha = keras.layers.MultiHeadAttention(n_heads, hidden_size)
      self.mlp1 = keras.layers.Dense(hidden_size, tf.nn.gelu)
      self.mlp2 = keras.layers.Dense(hidden_size, tf.nn.gelu)

    
    def call(self, inp):
       x = self.norm1(inp)
       x = self.mha(x, x)
       x = keras.layers.Add()([x, inp])

       x_1 = self.norm2(x)
       x_1 = self.mlp1(x_1)
       x_1 = self.mlp2(x_1)
       out = keras.layers.Add()([x, x_1])
       return out


In [21]:
tran_encoder = TransfomerEncoder(4, 768)
tran_encoder(tf.zeros([1, 256, 768]))

<tf.Tensor: shape=(1, 256, 768), dtype=float32, numpy=
array([[[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]]], dtype=float32)>

#### VIT Model

In [ ]:
class VIT(keras.models.Model):
    def __init__(self, n_layers, n_heads, patch_size, hidden_size, dense_1, dense_2, num_classes):
        super(VIT, self).__init__(name='vision_transformer')
        self.patch_encoder = PatchEncoder(patch_size, hidden_size)
        self.trans_encoders = [TransfomerEncoder(n_heads, hidden_size) for _ in range(n_layers)]

        # MLP Layer for classification
        self.dense_1 = keras.layers.Dense(dense_1, tf.nn.gelu)
        self.dense_2 = keras.layers.Dense(dense_2, tf.nn.gelu)
        self.dense_3 = keras.layers.Dense(num_classes, 'softmax')

    def call(self, x):
        x = self.patch_encoder(x)
        for tran_layer in self.trans_encoders:
            x = tran_layer(x)
        x = keras.layers.Flatten()(x)
        x = self.dense_1(x)
        x = self.dense_2(x)
        x = self.dense_3(x)
        return x


In [25]:
vit = VIT(5, 4, 16, 768, 512, 512, 3)
vit(tf.zeros([1, 256, 256, 3]))

<tf.Tensor: shape=(1, 3), dtype=float32, numpy=array([[0.13769008, 0.22210702, 0.64020294]], dtype=float32)>

In [26]:
vit.summary()

Model: "vision_transformer"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 patch_encoder (PatchEncode  multiple                  787200    
 r)                                                              
                                                                 
 transformer_encoder (Trans  multiple                  10631424  
 fomerEncoder)                                                   
                                                                 
 transformer_encoder (Trans  multiple                  10631424  
 fomerEncoder)                                                   
                                                                 
 transformer_encoder (Trans  multiple                  10631424  
 fomerEncoder)                                                   
                                                                 
 transformer_encoder (Trans  multiple           